## Import Libraries

In [37]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

## Import Dataset

In [38]:
df = pd.read_csv('../data/hypothyroid.csv')

In [39]:
print("Dataset Shape:", df.shape)
df.head(20)

Dataset Shape: (3772, 30)


,age,sex,on thyroxine,query on thyroxine,on antithyroid medication,sick,pregnant,thyroid surgery,I131 treatment,query hypothyroid,...,TT4 measured,TT4,T4U measured,T4U,FTI measured,FTI,TBG measured,TBG,referral source,binaryClass
0,41,F,f,f,f,f,f,f,f,f,...,t,125,t,1.14,t,109,f,?,SVHC,P
1,23,F,f,f,f,f,f,f,f,f,...,t,102,f,?,f,?,f,?,other,P
2,46,M,f,f,f,f,f,f,f,f,...,t,109,t,0.91,t,120,f,?,other,P
3,70,F,t,f,f,f,f,f,f,f,...,t,175,f,?,f,?,f,?,other,P
4,70,F,f,f,f,f,f,f,f,f,...,t,61,t,0.87,t,70,f,?,SVI,P
5,18,F,t,f,f,f,f,f,f,f,...,t,183,t,1.3,t,141,f,?,other,P
6,59,F,f,f,f,f,f,f,f,f,...,t,72,t,0.92,t,78,f,?,other,P
7,80,F,f,f,f,f,f,f,f,f,...,t,80,t,0.7,t,115,f,?,SVI,P
8,66,F,f,f,f,f,f,f,f,f,...,t,123,t,0.93,t,132,f,?,SVI,P
9,68,M,f,f,f,f,f,f,f,f,...,t,83,t,0.89,t,93,f,?,SVI,P


In [40]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3772 entries, 0 to 3771
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   age                        3772 non-null   str  
 1   sex                        3772 non-null   str  
 2   on thyroxine               3772 non-null   str  
 3   query on thyroxine         3772 non-null   str  
 4   on antithyroid medication  3772 non-null   str  
 5   sick                       3772 non-null   str  
 6   pregnant                   3772 non-null   str  
 7   thyroid surgery            3772 non-null   str  
 8   I131 treatment             3772 non-null   str  
 9   query hypothyroid          3772 non-null   str  
 10  query hyperthyroid         3772 non-null   str  
 11  lithium                    3772 non-null   str  
 12  goitre                     3772 non-null   str  
 13  tumor                      3772 non-null   str  
 14  hypopituitary              3772 non

| Column Name                   | Description                                                                                                                               |
| ----------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------- |
| **age**                       | Patient’s age, likely in years. Numeric but stored as an object due to inconsistent formatting or missing values.                         |
| **sex**                       | Gender of the patient (e.g., “M” for male, “F” for female).                                                                               |
| **on thyroxine**              | Whether the patient is taking thyroxine (a hormone replacement medication used in hypothyroidism). Values like “t” (true) or “f” (false). |
| **query on thyroxine**        | Indicates if there is a query about whether the patient should be on thyroxine treatment.                                                 |
| **on antithyroid medication** | Whether the patient is on antithyroid medication (used for hyperthyroidism treatment).                                                    |
| **sick**                      | Whether the patient is generally sick or suffering from other illnesses at the time of measurement.                                       |
| **pregnant**                  | Whether the patient is pregnant at the time of measurement.                                                                               |
| **thyroid surgery**           | History of thyroid surgery in the patient.                                                                                                |
| **I131 treatment**            | Whether the patient has undergone radioactive iodine treatment, commonly used to treat hyperthyroidism or thyroid cancer.                 |
| **query hypothyroid**         | Whether there is suspicion or query about hypothyroidism (underactive thyroid).                                                           |
| **query hyperthyroid**        | Whether there is suspicion or query about hyperthyroidism (overactive thyroid).                                                           |
| **lithium**                   | Whether the patient is on lithium medication, which can interfere with thyroid function.                                                  |
| **goitre**                    | Presence of goitre (enlargement of the thyroid gland).                                                                                    |
| **tumor**                     | Presence of tumor affecting the thyroid gland.                                                                                            |
| **hypopituitary**             | Indicates issues with the pituitary gland, which affects thyroid function.                                                                |
| **psych**                     | Whether the patient has psychiatric conditions, which can be associated with thyroid disorders.                                           |
| **TSH measured**              | Whether the Thyroid-Stimulating Hormone (TSH) test was measured during the visit.                                                         |
| **TSH**                       | Thyroid-Stimulating Hormone level; a key indicator of thyroid function.                                                                   |
| **T3 measured**               | Whether the Triiodothyronine (T3) hormone was measured.                                                                                   |
| **T3**                        | Level of T3 hormone in the blood; high or low levels can indicate thyroid disease.                                                        |
| **TT4 measured**              | Whether the Total Thyroxine (TT4) test was performed.                                                                                     |
| **TT4**                       | Level of total thyroxine hormone, another important thyroid hormone measurement.                                                          |
| **T4U measured**              | Whether the T4 Uptake test was performed.                                                                                                 |
| **T4U**                       | Results of the T4 uptake test, helping assess thyroid function.                                                                           |
| **FTI measured**              | Whether the Free Thyroxine Index (FTI) was measured.                                                                                      |
| **FTI**                       | A calculated measure combining TT4 and T4U to assess thyroid hormone availability.                                                        |
| **TBG measured**              | Whether Thyroxine-Binding Globulin (TBG) levels were measured.                                                                            |
| **TBG**                       | Level of protein that carries thyroid hormones in the bloodstream.                                                                        |
| **referral source**           | The source through which the patient was referred for thyroid testing (e.g., other clinics, hospitals, specialists).                      |
| **binaryClass**               | Target variable; likely indicates whether the patient has thyroid disease (“1” or “positive”) or not (“0” or                                            “negative”).

## Data Cleaning & Preproccesing

#### **_Replace '?' with NaN_**

In [41]:
df = df.replace('?', np.nan)

#### **_Convert target_**

In [77]:
print(df['binaryClass'].value_counts())

binaryClass
P    3481
N     291
Name: count, dtype: int64


In [42]:
df["binaryClass"] = df["binaryClass"].map({"P":1 , "N":0})

In [43]:
print(df['binaryClass'].value_counts())

binaryClass
1    3481
0     291
Name: count, dtype: int64


#### **_Convert f/t columns_**

In [44]:
binary_columns = [
    'on thyroxine',
    'query on thyroxine',
    'on antithyroid medication',
    'sick',
    'pregnant',
    'thyroid surgery',
    'I131 treatment',
    'query hypothyroid',
    'query hyperthyroid',
    'lithium',
    'goitre',
    'tumor',
    'hypopituitary',
    'psych',
    'TSH measured',
    'T3 measured',
    'TT4 measured',
    'T4U measured',
    'FTI measured',
    'TBG measured'
]

for column in binary_columns:
    df[column] = df[column].map({
        'f': 0,
        't': 1
    })

#### **_Convert sex_**

In [45]:
df['sex'] = df['sex'].map({
    'F': 1,
    'M': 0
})

#### **_Fill missing sex_**
# Fill the 150 missing sex values using the training process later.
# For now, leave them as NaN.

In [88]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3772 entries, 0 to 3771
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   age                        3771 non-null   str  
 1   sex                        3772 non-null   int64
 2   on thyroxine               3772 non-null   int64
 3   query on thyroxine         3772 non-null   int64
 4   on antithyroid medication  3772 non-null   int64
 5   sick                       3772 non-null   int64
 6   pregnant                   3772 non-null   int64
 7   thyroid surgery            3772 non-null   int64
 8   I131 treatment             3772 non-null   int64
 9   query hypothyroid          3772 non-null   int64
 10  query hyperthyroid         3772 non-null   int64
 11  lithium                    3772 non-null   int64
 12  goitre                     3772 non-null   int64
 13  tumor                      3772 non-null   int64
 14  hypopituitary              3772 non

#### **_Convert numerical columns from string to numeric_**

In [46]:
# Convert numerical columns from string to numeric
numeric_columns = [
    'age',
    'TSH',
    'T3',
    'TT4',
    'T4U',
    'FTI'
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors='coerce'
    )

In [47]:
print("Cleaning completed.")

print("\nImportant columns:")
print(df[['age', 'sex', 'on thyroxine',
          'TSH', 'T3 measured', 'T3', 'TT4']].head())

print("\nMissing values:")
print(df[['age', 'sex', 'on thyroxine',
          'TSH', 'T3 measured', 'T3', 'TT4']].isnull().sum())

Cleaning completed.

Important columns:
    age  sex  on thyroxine   TSH  T3 measured   T3    TT4
0  41.0  1.0             0  1.30            1  2.5  125.0
1  23.0  1.0             0  4.10            1  2.0  102.0
2  46.0  0.0             0  0.98            0  NaN  109.0
3  70.0  1.0             1  0.16            1  1.9  175.0
4  70.0  1.0             0  0.72            1  1.2   61.0

Missing values:
age               1
sex             150
on thyroxine      0
TSH             369
T3 measured       0
T3              769
TT4             231
dtype: int64


In [92]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3772 entries, 0 to 3771
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        3771 non-null   float64
 1   sex                        3772 non-null   int64  
 2   on thyroxine               3772 non-null   int64  
 3   query on thyroxine         3772 non-null   int64  
 4   on antithyroid medication  3772 non-null   int64  
 5   sick                       3772 non-null   int64  
 6   pregnant                   3772 non-null   int64  
 7   thyroid surgery            3772 non-null   int64  
 8   I131 treatment             3772 non-null   int64  
 9   query hypothyroid          3772 non-null   int64  
 10  query hyperthyroid         3772 non-null   int64  
 11  lithium                    3772 non-null   int64  
 12  goitre                     3772 non-null   int64  
 13  tumor                      3772 non-null   int64  
 14  hyp

#### **_Remove TBG and referral source (We dont need it)_**

In [93]:
df = df.drop(columns=['TBG', 'referral source'])

In [95]:
print(df.shape)
df.info()

(3772, 28)
<class 'pandas.DataFrame'>
RangeIndex: 3772 entries, 0 to 3771
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        3771 non-null   float64
 1   sex                        3772 non-null   int64  
 2   on thyroxine               3772 non-null   int64  
 3   query on thyroxine         3772 non-null   int64  
 4   on antithyroid medication  3772 non-null   int64  
 5   sick                       3772 non-null   int64  
 6   pregnant                   3772 non-null   int64  
 7   thyroid surgery            3772 non-null   int64  
 8   I131 treatment             3772 non-null   int64  
 9   query hypothyroid          3772 non-null   int64  
 10  query hyperthyroid         3772 non-null   int64  
 11  lithium                    3772 non-null   int64  
 12  goitre                     3772 non-null   int64  
 13  tumor                      3772 non-null   int64

**_Check missing values_**

In [96]:
print(df.isnull().sum())

age                            1
sex                            0
on thyroxine                   0
query on thyroxine             0
on antithyroid medication      0
sick                           0
pregnant                       0
thyroid surgery                0
I131 treatment                 0
query hypothyroid              0
query hyperthyroid             0
lithium                        0
goitre                         0
tumor                          0
hypopituitary                  0
psych                          0
TSH measured                   0
TSH                          369
T3 measured                    0
T3                           769
TT4 measured                   0
TT4                          231
T4U measured                   0
T4U                          387
FTI measured                   0
FTI                          385
TBG measured                   0
binaryClass                    0
dtype: int64


**_Handle missing values_**

In [98]:
print(df.isnull().sum())

age                          0
sex                          0
on thyroxine                 0
query on thyroxine           0
on antithyroid medication    0
sick                         0
pregnant                     0
thyroid surgery              0
I131 treatment               0
query hypothyroid            0
query hyperthyroid           0
lithium                      0
goitre                       0
tumor                        0
hypopituitary                0
psych                        0
TSH measured                 0
TSH                          0
T3 measured                  0
T3                           0
TT4 measured                 0
TT4                          0
T4U measured                 0
T4U                          0
FTI measured                 0
FTI                          0
TBG measured                 0
binaryClass                  0
dtype: int64


## _Spliting the Target and Test Variable_

## _Select the 7 features and target_

In [48]:
features = [
    'age',
    'sex',
    'on thyroxine',
    'TSH',
    'T3 measured',
    'T3',
    'TT4'
]

X = df[features].copy()
y = df['binaryClass'].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget distribution:")
print(y.value_counts())

X shape: (3772, 7)
y shape: (3772,)

Features:
['age', 'sex', 'on thyroxine', 'TSH', 'T3 measured', 'T3', 'TT4']

Target distribution:
binaryClass
1    3481
0     291
Name: count, dtype: int64


## _Train/Test Split_

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_test distribution:")
print(y_test.value_counts())

X_train shape: (3017, 7)
X_test shape: (755, 7)

y_train distribution:
binaryClass
1    2784
0     233
Name: count, dtype: int64

y_test distribution:
binaryClass
1    697
0     58
Name: count, dtype: int64


## _Build the final Thyroid Pipeline_

In [50]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

thyroid_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])

# Train the complete pipeline
thyroid_model.fit(X_train, y_train)

print("Final Thyroid Pipeline trained successfully.")

Final Thyroid Pipeline trained successfully.


## _Evaluate the new pipeline_

In [51]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Predictions on test data
y_test_pred = thyroid_model.predict(X_test)

# Accuracy
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Test Accuracy:", test_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

Test Accuracy: 0.9788079470198675

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.98      0.88        58
           1       1.00      0.98      0.99       697

    accuracy                           0.98       755
   macro avg       0.90      0.98      0.93       755
weighted avg       0.98      0.98      0.98       755


Confusion Matrix:
[[ 57   1]
 [ 15 682]]


## _Check training performance_

In [52]:
# Training predictions
y_train_pred = thyroid_model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

print("Training Accuracy:", train_accuracy)
print("Testing Accuracy:", test_accuracy)

print("\nTraining Classification Report:")
print(classification_report(y_train, y_train_pred))

Training Accuracy: 0.9761352336758369
Testing Accuracy: 0.9788079470198675

Training Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.97      0.86       233
           1       1.00      0.98      0.99      2784

    accuracy                           0.98      3017
   macro avg       0.89      0.97      0.92      3017
weighted avg       0.98      0.98      0.98      3017



## _Save the corrected model_

In [53]:
import pickle

model_path = '../models/Thyroid_model.sav'

with open(model_path, 'wb') as file:
    pickle.dump(thyroid_model, file)

print("Corrected Thyroid model saved successfully.")

Corrected Thyroid model saved successfully.


In [54]:
import os

print("File exists:", os.path.exists(model_path))

File exists: True


## _Reload and inspect the saved model_

In [55]:
import pickle

with open('../models/Thyroid_model.sav', 'rb') as file:
    loaded_thyroid_model = pickle.load(file)

print("Saved model loaded successfully.")
print(type(loaded_thyroid_model))

print("\nImputer statistics:")
print(loaded_thyroid_model.named_steps['imputer'].statistics_)

print("\nNumber of features:")
print(loaded_thyroid_model.named_steps['imputer'].n_features_in_)

Saved model loaded successfully.
<class 'sklearn.pipeline.Pipeline'>

Imputer statistics:
[ 54.    1.    0.    1.4   1.    2.  104. ]

Number of features:
7


## _Final real-patient-style test_

In [56]:
import pandas as pd

new_patient = pd.DataFrame([{
    'age': 44,
    'sex': 0,
    'on thyroxine': 0,
    'TSH': 45.0,
    'T3 measured': 1,
    'T3': 1.4,
    'TT4': 39.0
}])

prediction = loaded_thyroid_model.predict(new_patient)

print("Prediction:", prediction[0])

probability = loaded_thyroid_model.predict_proba(new_patient)

print("\nPrediction probabilities:")
print("Class 0 (N):", probability[0][0])
print("Class 1 (P):", probability[0][1])

Prediction: 0

Prediction probabilities:
Class 0 (N): 0.9999999997814374
Class 1 (P): 2.1856263932296432e-10
